# 01 — Phân tích Dữ liệu Toàn diện (Gộp Trước & Sau Tiền Xử Lý)

Khảo sát, kiểm định chất lượng và phân tích thống kê bộ dữ liệu phát hiện biển báo giao thông Việt Nam (VNTS - 52 lớp).  
Owner: **Team analysis + preprocessing offline**.

---

### Cấu trúc Luồng Phân tích Hợp nhất (Consolidated Flow):
1. **Phần I — Phân tích Dữ liệu Gốc (`data/raw`) trên TOÀN BỘ ẢNH (Không dùng split mặc định)**:
   - Khảo sát toàn diện 3,216 ảnh gốc và 8,334 bounding box.
   - **Bỏ biểu đồ 05 và 07 cũ** (vì không dùng split mặc định bị rò rỉ).
   - **Bổ sung Figure 05: Phân tích Gom cụm dHash tương đồng** (`hash_size=32`, `hamming <= 370`) tìm ra **401 cụm độc lập** (phát hiện chuỗi frame video dashcam phục vụ chia lại split an toàn).
   - Bộ **8 biểu đồ chuẩn** từ `01` đến `08`.

2. **Phần II — Phân tích Dữ liệu Sau Tiền Xử Lý (`data/processed`)**:
   - Đánh giá lại sau khi chạy pipeline tiền xử lý (lọc tiny box < 8px, letterbox 640x640, và phân chia split mới).
   - Kiểm định tỷ lệ box/split và đo kiểm rò rỉ dữ liệu dHash giữa các split mới.
   - Bộ **9 biểu đồ chuẩn** từ `01` đến `09`.

3. **Phần III — Bảng So sánh Đối chiếu Trước vs Sau Tiền Xử Lý & Đề xuất Re-split**:
   - Tổng hợp số liệu so sánh Before vs After và hướng dẫn chia lại tập train/val/test theo cụm (GroupKFold).

## 0) Bootstrap Môi trường

Tự động chuẩn bị repo trên Local, Colab hoặc Kaggle.

In [ ]:
import os, sys, subprocess
from pathlib import Path

# ── Tìm repo root ──────────────────────────────────────────────
REPO_URL    = 'https://github.com/tuan8p/VN-Traffic-Sign-Detection.git'
REPO_DIR    = 'VN-Traffic-Sign-Detection'
REPO_BRANCH = 'preprocessing'

base = Path('/kaggle/working') if Path('/kaggle/working').exists() else Path.cwd()
repo_path = base / REPO_DIR

def sh(cmd):
    print('>>', cmd)
    subprocess.run(cmd, shell=True, check=True)

if (Path.cwd() / '.git').exists() and (Path.cwd() / 'configs' / 'shared.yaml').exists():
    repo_path = Path.cwd()
    print('Using the repo this notebook sits in:', repo_path)
elif not (repo_path / '.git').exists():
    sh(f'git clone --branch {REPO_BRANCH} {REPO_URL} "{repo_path}"')
    git = f'git -C "{repo_path}"'
    sh(f'{git} fetch origin {REPO_BRANCH}')
    sh(f'{git} checkout -B {REPO_BRANCH} origin/{REPO_BRANCH}')

os.chdir(repo_path)
if str(repo_path) not in sys.path:
    sys.path.insert(0, str(repo_path))
print('cwd:', Path.cwd())
print('Bootstrap OK.')

## 1) Helper Function Hiển thị Biểu đồ & Tóm tắt Thống kê

In [ ]:
import json
from pathlib import Path
import matplotlib.pyplot as plt
import matplotlib.image as mpimg

def show_figure(fig_dir: Path, prefix: str, title: str):
    matches = sorted(fig_dir.glob(f"{prefix}*.png"))
    if matches:
        img = mpimg.imread(matches[0])
        plt.figure(figsize=(18, 10))
        plt.imshow(img)
        plt.axis('off')
        plt.title(f"{title} ({matches[0].name})", fontsize=12, fontweight='bold', pad=10)
        plt.tight_layout()
        plt.show()
    else:
        print(f"[-] Chưa tìm thấy biểu đồ tiền tố: {prefix} trong {fig_dir}")

def print_summary_report(fig_dir: Path, title: str):
    summary_files = sorted(fig_dir.glob('eda_summary*.json'))
    if not summary_files:
        print(f"[-] Không tìm thấy file summary json trong {fig_dir}")
        return
    with open(summary_files[0], 'r', encoding='utf-8') as f:
        meta = json.load(f)
    print(f'==================== BÁO CÁO TÓM TẮT THỐNG KÊ: {title.upper()} ====================')
    print(f"• Tổng số ảnh              : {meta.get('n_images', 'N/A'):,}")
    print(f"• Tổng số bounding box     : {meta.get('n_boxes', 'N/A'):,}")
    if 'splits' in meta:
        print(f"• Phân bố Splits           : {meta.get('splits')}")
    if 'clustering' in meta:
        cl = meta['clustering']
        print(f"• Phân tích Gom Cụm dHash  : Tổng {cl.get('total_clusters', 0):,} cụm độc lập")
        print(f"  - Cụm đơn lẻ (Singletons): {cl.get('singleton_clusters', 0):,} ảnh ({cl.get('singleton_pct', 0)}%)")
        print(f"  - Cụm đa ảnh (Near-dup)  : {cl.get('multi_image_clusters', 0):,} cụm ({cl.get('multi_image_total_images', 0):,} ảnh, {cl.get('multi_image_pct', 0)}%)")
        print(f"  - Cụm lớn nhất           : {cl.get('max_cluster_size', 0)} ảnh/chuỗi")
    if 'leakage' in meta:
        lk = meta['leakage']
        print(f"• Trạng thái rò rỉ (dHash) : {lk.get('status', 'N/A')} (Tổng số cặp rò rỉ: {lk.get('total_leak_pairs', 0)})")
        for sp, info in lk.get('leak_summary_by_split', {}).items():
            print(f"  - Cặp split {sp:<14}: {info.get('leak_pairs_count', 0)} cặp rò rỉ")
    if 'quality' in meta:
        q = meta['quality']
        print(f"• Độ sáng trung bình       : {q.get('brightness_mean', 0):.1f} / 255 (Tối: {q.get('dark_pct', 0):.1f}%, Chói: {q.get('bright_pct', 0):.1f}%)")
        print(f"• Độ nét trung vị          : {q.get('sharpness_median', 0):.1f} (Tỷ lệ ảnh mờ: {q.get('blur_pct', 0):.1f}%)")
    print('==========================================================================\n')

## PHẦN I — Khảo sát & Phân tích Dữ liệu Gốc (`data/raw`)

Phân tích toàn diện trên **toàn bộ 3,216 ảnh gốc ban đầu** (bỏ qua split mặc định cũ do bị rò rỉ 54% test sang train).  
Bộ **8 biểu đồ chuẩn từ 01 đến 08** (kèm Figure 05: gom cụm dHash size=32, hamming <= 370).

In [ ]:
from analysis.eda import main as eda_main

# Cấu hình dHash size và ngưỡng Hamming:
# - dsize=16 (256 bit)  -> hamming=24 (~9.4%)
# - dsize=24 (576 bit)  -> hamming=54 (~9.4%)
# - dsize=32 (1024 bit) -> hamming=96 (~9.4%)
DHASH_SIZE    = 32
DHASH_HAMMING = 370

print("[1/2] Đang chạy phân tích toàn diện trên DATA/RAW...")
eda_main([
    '--data-root',     'data/raw',
    '--figures-dir',   'analysis/figures/raw',
    '--dhash-size',    str(DHASH_SIZE),
    '--dhash-hamming', str(DHASH_HAMMING),
])


In [ ]:
raw_fig_dir = Path('analysis/figures/raw')

raw_figures = [
    ("01", "01. Phân bố 52 lớp biển báo (Raw toàn bộ)"),
    ("02", "02. Phân bố số lượng box trên mỗi ảnh"),
    ("03", "03. Phân bố kích thước bounding box"),
    ("04", "04. Mật độ không gian tâm box"),
    ("05", "05. Phân tích Gom cụm dHash Tương đồng (size=32, Hamming <= 370)"),
    ("06", "06. Đánh giá Tiny Box (< 8px)"),
    ("07", "07. Độ sáng & Độ sắc nét quang học"),
    ("08", "08. Ảnh mẫu kèm nhãn tiếng Việt"),
]

for pfx, title in raw_figures:
    show_figure(raw_fig_dir, pfx, title)

print_summary_report(raw_fig_dir, "Dữ liệu Gốc (DATA/RAW)")

## PHẦN II — Phân tích & Kiểm định Dữ liệu Sau Tiền Xử Lý (`data/processed`)

Đánh giá dữ liệu sau khi đã tiền xử lý offline (loại bỏ tiny box < 8px, letterbox 640x640 và phân chia split mới).  
Bộ **9 biểu đồ chuẩn từ 01 đến 09** (kèm kiểm tra phân bố split mới và đo kiểm rò rỉ bối cảnh).

In [ ]:
print("[2/2] Đang chạy phân tích kiểm định trên DATA/PROCESSED...")
eda_main([
    '--data-root', 'data/processed',
    '--figures-dir', 'analysis/figures/processed',
])

In [ ]:
proc_fig_dir = Path('analysis/figures/processed')

proc_figures = [
    ("01", "01. Phân bố 52 lớp biển báo (Processed)"),
    ("02", "02. Phân bố số lượng box trên mỗi ảnh"),
    ("03", "03. Phân bố kích thước bounding box"),
    ("04", "04. Mật độ không gian tâm box"),
    ("05", "05. Số lượng ảnh & box theo từng split mới"),
    ("06", "06. Đánh giá Tiny Box (Xác nhận 0% box < 8px)"),
    ("07", "07. Kiểm tra Rò rỉ Dữ liệu giữa các Split (dHash Overlap & Hamming Distance)"),
    ("08", "08. Độ sáng & Độ sắc nét quang học"),
    ("09", "09. Ảnh mẫu kèm nhãn tiếng Việt"),
]

for pfx, title in proc_figures:
    show_figure(proc_fig_dir, pfx, title)

print_summary_report(proc_fig_dir, "Dữ liệu Sau Tiền Xử Lý (DATA/PROCESSED)")

## PHẦN III — Bảng So sánh Đối chiếu Trước vs Sau Tiền Xử Lý

| Tiêu chí | Trước tiền xử lý (`data/raw`) | Sau tiền xử lý (`data/processed`) | Ý nghĩa kỹ thuật cho Báo cáo BTL |
| :--- | :--- | :--- | :--- |
| **Phạm vi phân tích** | Toàn bộ 3,216 ảnh gốc (toàn cục) | 3,851 ảnh (2,799 Train / 539 Val / 513 Test) | Phân chia độc lập theo cụm dHash và tăng cường tập huấn luyện. |
| **Tổng số Bounding Box** | 8,334 box (52/52 lớp xuất hiện) | 9,895 box (52/52 lớp xuất hiện) | Bảo toàn đầy đủ 52 lớp biển báo; tăng cường 2,372 box cho các lớp thiểu số trong Train. |
| **Phân bố Lớp (Figure 01)** | Cột đứng 52 lớp giảm dần | 3 Subplot riêng biệt (Train Real vs Aug, Val, Test) | Quan sát trực quan phân bố trên từng tập; phân biệt rõ ảnh thật (5,221 box) và ảnh tự tạo (2,372 box). |
| **Cân bằng Nhãn (Imbalance)** | Chênh lệch lớn: Max=1,071, Min=3 box | Cải thiện mạnh: Min tăng lên 21 box, Median tăng từ 53 lên 85 box | Giảm thiểu thiên lệch (bias) dự đoán đối với các biển báo hiếm. |
| **Tiny Box (< 8px)** | **811 box (9.73%)** | **0 box (0.00%)** | Triệt tiêu hoàn toàn tiny box dưới ngưỡng anchor, chống gradient bất ổn định. |
| **Phân tích Cụm / Data Leakage** | **390 cụm dHash** (size=32, Hamming $\le$ 370). Bóc tách viền xám 114 khi tính hash | **Khảo sát chuẩn 1024-bit (size=32, Hamming $\le$ 370)** trên vùng ảnh thực tế | Phân chia tập theo cụm giúp triệt tiêu hiện tượng học vẹt chuỗi video; loại bỏ viền xám đệm để tránh nhận nhầm tương đồng. |
| **Cân bằng Quang học (Sáng / Mờ)** | 159 ảnh tối, 22 ảnh mờ tập trung cục bộ | Phân bổ đều giữa các tập (Train: 120 tối, 19 mờ; Val: 23 tối, 1 mờ; Test: 16 tối, 2 mờ) | Đảm bảo các tập không bị lệch quang học, đánh giá khách quan. |
| **Kích thước canvas** | Đa độ phân giải (chuẩn 16:9 gốc) | Letterbox 640x640 chuẩn hóa | Tối ưu hóa kích thước đầu vào đồng nhất cho Faster R-CNN và YOLOv11. |

### Đánh giá Kết quả Preprocessing & EDA:
1. **Phân chia Biểu đồ 01 Rõ ràng**: Tách thành 3 subplot riêng cho Train, Val, Test. Trên tập Train, sử dụng biểu đồ cột chồng phân định rõ Ảnh Thật (`#1D3557`) và Ảnh Tự tạo (`#E76F51`), không còn chữ kết luận rườm rà.
2. **Khảo sát Rò rỉ Chuẩn xác (Bỏ Viền Xám)**: Khi tính mã dHash, thuật toán tự động bóc tách dải màu xám đệm (114) sinh ra do letterbox, đảm bảo chỉ so sánh nội dung cảnh thực.
3. **Cấu trúc Module Hóa Cao**: Toàn bộ logic phân tích sau tiền xử lý được chuyển về thư mục `analysis/` (`image_utils.py`, `leakage.py`, `clustering.py`, `class_distribution.py`), giúp mã nguồn gọn gàng, tách bạch và dễ bảo trì.
